# Chapter 20 — Mixture of Experts

Every claim in Chapter 20, recomputed from `tfs/moe.py`. Each cell **asserts** what it checks,
so `python tests/run_notebooks.py 20` fails if the chapter and the code ever disagree.

In [ ]:
import math, sys, os
sys.path.insert(0, os.path.abspath(".."))      # run from notebooks/
sys.path.insert(0, os.path.abspath("."))       # or from the repo root
import torch
from tfs import moe as M
torch.manual_seed(0)

## 20.2 — one token through the router (the worked example)

In [ ]:
logits = torch.tensor([[0.1, 2.4, -0.3, 1.9, 0.5, -1.2, 0.8, 0.2]])   # (N=1, E=8)
w, idx, p = M.route_softmax_topk(logits, 2)
print("probs", [round(v, 4) for v in p[0].tolist()])
print("chosen", idx[0].tolist(), "gates", [round(v, 4) for v in w[0].tolist()])
assert idx[0].tolist() == [1, 3]
assert [round(v, 4) for v in w[0].tolist()] == [0.6225, 0.3775]
# Mixtral's form gives the same answer
w2, idx2, _ = M.route_topk_softmax(logits, 2)
assert torch.equal(idx, idx2) and torch.allclose(w, w2)

## 20.2 — top-1 with renormalisation: the router never learns

In [ ]:
lg = torch.randn(32, 8, requires_grad=True)
w, _, _ = M.route_topk_softmax(lg, 1)
(w * torch.randn_like(w)).sum().backward()
print("grad with gate == 1:", lg.grad.abs().max().item())
assert lg.grad.abs().max() == 0
lg.grad = None
w, _, _ = M.route_softmax_topk(lg, 1, renorm=False)                # Switch keeps p
(w * torch.randn_like(w)).sum().backward()
print("grad with gate == p:", lg.grad.abs().max().item())
assert lg.grad.abs().max() > 1e-3

## 20.3 — the sorted (grouped-GEMM) layout computes exactly what the loop computes

In [ ]:
x = torch.randn(2, 16, 32)                                          # (B, T, C)
for g in ("softmax", "topk_softmax", "sigmoid"):
    m = M.MoE(32, 48, n_expert=8, top_k=2, n_shared=1, gating=g)
    a, _ = m.forward_loop(x)
    b, _ = m(x)
    assert torch.allclose(a, b, atol=1e-5), g
print("loop == sorted dispatch for all three routers")

## 20.3 — capacity factor and dropped tokens (64 experts, top-2, 4096 tokens)

In [ ]:
def drop_rate(cf, skew, E=64, k=2, N=4096, trials=50):
    tot = 0.0
    for _ in range(trials):
        lg = torch.randn(N, E) + skew * torch.linspace(1, -1, E)
        idx = lg.topk(k, -1).indices
        cap = M.expert_capacity(N, E, k, cf)
        tot += (M.expert_load(idx, E) - cap).clamp(min=0).sum().item() / (N * k)
    return 100 * tot / trials
torch.manual_seed(0)
for cf in (1.0, 1.25, 1.5, 2.0):
    print(f"cf {cf:4}:  uniform {drop_rate(cf, 0.0):5.2f}%   mild skew {drop_rate(cf, 0.5):5.2f}%")
assert M.expert_capacity(4096, 64, 2, 1.25) == 160

## 20.4 — the auxiliary loss: 1 when balanced, E when collapsed

In [ ]:
E, N = 8, 800
uni = M.load_balance_loss(torch.full((N, E), 1 / E), torch.arange(N).remainder(E)[:, None])
col = torch.zeros(N, E); col[:, 3] = 1
dead = M.load_balance_loss(col, torch.full((N, 1), 3))
part = M.load_balance_loss(torch.tensor([[0.7, 0.1, 0.1, 0.1]]).repeat(100, 1), torch.zeros(100, 1, dtype=torch.long))
print(round(uni.item(), 4), round(dead.item(), 4), round(part.item(), 4))
assert abs(uni.item() - 1) < 1e-6 and abs(dead.item() - E) < 1e-6 and abs(part.item() - 2.8) < 1e-5

## 20.4 — DeepSeek-V3's bias rule balances a skewed router and never touches the gates

In [ ]:
torch.manual_seed(1)
E, k, N = 16, 2, 4096
skew, bias = torch.linspace(2.0, -2.0, E), torch.zeros(E)
history = []
for step in range(400):
    lg = torch.randn(N, E) + skew
    w, idx, _ = M.route_sigmoid_bias(lg, k, bias)
    history.append(M.load_cv(idx, E))
    M.update_bias(bias, idx, gamma=0.02)
print("load CV: step 0 %.2f  step 50 %.2f  step 399 %.2f" % (history[0], history[50], history[-1]))
assert history[0] > 1.0 and history[-1] < 0.15
s = lg.sigmoid(); g = s.gather(-1, idx); g = g / g.sum(-1, keepdim=True)
assert torch.allclose(w, g)          # gates are the unbiased affinities

## 20.5 — fine-grained experts: combinations

In [ ]:
for E, k in ((8, 2), (32, 8), (256, 8)):
    print(f"{E:4} experts, top-{k}: {math.comb(E, k):.3e} combinations")
assert math.comb(8, 2) == 28 and math.comb(32, 8) == 10_518_300
assert 4.0e14 < math.comb(256, 8) < 4.2e14

## 20.1 / 20.9 — parameter arithmetic

In [ ]:
tot, act = M.moe_params(7168, 2048, 256, 8, n_shared=1)            # one DeepSeek-V3 MoE layer
print(f"DeepSeek-V3 MoE layer: {tot/1e9:.2f} B total, {act/1e9:.3f} B active; x58 layers: {58*tot/1e9:.0f} B, {58*act/1e9:.1f} B")
assert round(58 * tot / 1e9) == 657 and round(58 * act / 1e9, 1) == 23.1
per = M.mlp_params(4096, 14336)                                     # one Mixtral expert
assert round(8 * per * 32 / 1e9, 1) == 45.1 and round(2 * per * 32 / 1e9, 1) == 11.3

## 20.6 — the all-to-all bill (DeepSeek-V3 shape, 4096 tokens per device)

In [ ]:
N, k, C, layers = 4096, 8, 7168, 58
disp, comb = N * k * C * 1, N * k * C * 2                           # fp8 dispatch, bf16 combine
per_layer = disp + comb
print(f"per layer {per_layer/1e9:.2f} GB, per forward {layers*per_layer/1e9:.1f} GB")
print(f"over NVLink (900 GB/s) {layers*per_layer/900e9*1e3:.0f} ms, over InfiniBand (50 GB/s) {layers*per_layer/50e9*1e3:.0f} ms")
assert round(per_layer / 1e9, 2) == 0.70

## 20.7 — how much of the model a decode step reads

In [ ]:
rows = {"Mixtral (8, top-2)": (8, 2), "DeepSeek-V3 (256, top-8)": (256, 8),
        "Qwen3-235B (128, top-8)": (128, 8), "gpt-oss-120b (128, top-4)": (128, 4)}
for name, (E, k) in rows.items():
    print(f"{name:28}", "  ".join(f"b={b}: {100*M.expected_distinct_experts(E, k, b)/E:3.0f}%" for b in (1, 8, 32, 64, 256)))
assert round(M.expected_distinct_experts(256, 8, 64)) == 222
assert round(100 * M.expected_distinct_experts(8, 2, 8) / 8) == 90
# and the formula agrees with simulation
sims = [torch.rand(64, 256).argsort(-1)[:, :8].unique().numel() for _ in range(300)]
assert abs(sum(sims) / len(sims) - M.expected_distinct_experts(256, 8, 64)) < 1.0